In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("C:\AIML\Car Resale price prediction with Chatbot Enhanced Recommendations\Cleaned_cars.csv")
df.head()

,fuel_type,body_type,km_driven,transmission,owner_number,brand,model,model_year,variant,price_raw,...,rto,feature,mileage,engine,max_power,torque,seats,city,car_links,price_lakh
0,PETROL,Hatchback,120000,Manual,3,Maruti,Maruti Celerio,2015,VXI,₹ 4 Lakh,...,KA51,"Power Steering, Power Windows Front, Power Win...",23.10,998.0,67.04,90.00,5.0,bangalore,https://www.cardekho.com/used-car-details/used...,4.00
1,PETROL,SUV,32706,Manual,2,Ford,Ford Ecosport,2018,1.5 Petrol Titanium BSIV,₹ 8.11 Lakh,...,KA05,"Power Steering, Power Windows Front, Power Win...",17.00,1497.0,121.31,150.00,5.0,bangalore,https://www.cardekho.com/buy-used-car-details/...,8.11
2,PETROL,Hatchback,11949,Manual,1,Tata,Tata Tiago,2018,1.2 Revotron XZ,₹ 5.85 Lakh,...,KA03,"Power Steering, Power Windows Front, Power Win...",23.84,1199.0,84.00,114.00,5.0,bangalore,https://www.cardekho.com/used-car-details/used...,5.85
3,PETROL,Sedan,17794,Manual,1,Hyundai,Hyundai Xcent,2014,1.2 Kappa S Option,₹ 4.62 Lakh,...,KA53,"Power Steering, Power Windows Front, Power Win...",19.10,1197.0,81.86,113.75,5.0,bangalore,https://www.cardekho.com/buy-used-car-details/...,4.62
4,DIESEL,SUV,60000,Manual,1,Maruti,Maruti SX4 S Cross,2015,DDiS 200 Zeta,₹ 7.90 Lakh,...,KA04,"Power Steering, Power Windows Front, Power Win...",23.65,1248.0,88.50,200.00,5.0,bangalore,https://www.cardekho.com/used-car-details/used...,7.90


In [3]:
df.shape

(8239, 22)

In [4]:
features=[
    "brand",
    "model",
    "fuel_type",
    "body_type",
    "transmission",
    "city",
    "km_driven",
    "model_year",
    "mileage",
    "engine",
    "max_power",
    "price_lakh"
]

rec=df[features].copy()
rec.head()

,brand,model,fuel_type,body_type,transmission,city,km_driven,model_year,mileage,engine,max_power,price_lakh
0,Maruti,Maruti Celerio,PETROL,Hatchback,Manual,bangalore,120000,2015,23.10,998.0,67.04,4.00
1,Ford,Ford Ecosport,PETROL,SUV,Manual,bangalore,32706,2018,17.00,1497.0,121.31,8.11
2,Tata,Tata Tiago,PETROL,Hatchback,Manual,bangalore,11949,2018,23.84,1199.0,84.00,5.85
3,Hyundai,Hyundai Xcent,PETROL,Sedan,Manual,bangalore,17794,2014,19.10,1197.0,81.86,4.62
4,Maruti,Maruti SX4 S Cross,DIESEL,SUV,Manual,bangalore,60000,2015,23.65,1248.0,88.50,7.90


In [5]:
categorical=[
    "brand",
    "model",
    "fuel_type",
    "body_type",
    "transmission",
    "city"
]

categorical

['brand', 'model', 'fuel_type', 'body_type', 'transmission', 'city']

In [6]:
numerical=[
    "km_driven",
    "model_year",
    "mileage",
    "engine",
    "max_power",
    "price_lakh"
]

numerical

['km_driven', 'model_year', 'mileage', 'engine', 'max_power', 'price_lakh']

In [7]:
from sklearn.preprocessing import OneHotEncoder

encoder=OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

In [8]:
encoder_categorical=encoder.fit_transform(
    rec[categorical]
)

encoder_categorical.shape

(8239, 357)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

scaler_numerical=scaler.fit_transform(
    rec[numerical]
)

scaler_numerical.shape

(8239, 6)

In [10]:
matrix=np.hstack([
    encoder_categorical,
    scaler_numerical
])

matrix.shape

(8239, 363)

In [13]:
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

inertia = []
silhouette = []

for k in range(2, 9):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = kmeans.fit_predict(matrix)
    
    inertia.append(kmeans.inertia_)
    silhouette.append(
        silhouette_score(matrix, labels)
    )

print("K values:", list(range(2, 9)))
print("Inertia:", inertia)
print("Silhouette scores:", silhouette)

K values: [2, 3, 4, 5, 6, 7, 8]
Inertia: [66347.00475665116, 61005.40647599772, 55895.092983406204, 51702.89460888941, 48030.32572407598, 45410.92653446607, 43904.56879125241]
Silhouette scores: [0.3018074837434201, 0.2984569725277216, 0.11337553383921772, 0.12098873499723493, 0.11858123500609036, 0.12530781779956043, 0.12077134635304637]


In [14]:
final_kmeans=KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

final_clusters = final_kmeans.fit_predict(
    matrix
)

rec["cluster"] = final_clusters

rec["cluster"].value_counts().sort_index()

cluster
0    6509
1    1730
Name: count, dtype: int64

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_cars(
    budget,
    fuel,
    brand,
    body_type,
    transmission,
    city,
    top_n=10
):
    # --------------------------------------------------
    # 1. Filter by hard requirements
    # --------------------------------------------------
    candidates = rec[
        (rec["price_lakh"] <= budget) &
        (rec["fuel_type"].str.upper() == fuel.upper())
    ].copy()

    if candidates.empty:
        return pd.DataFrame()

    # --------------------------------------------------
    # 2. Create user preference vector
    # --------------------------------------------------
    user_preferences = pd.DataFrame([{
        "brand": brand,
        "model": "Unknown",
        "fuel_type": fuel,
        "body_type": body_type,
        "transmission": transmission,
        "city": city,
        "km_driven": rec["km_driven"].median(),
        "model_year": rec["model_year"].median(),
        "mileage": rec["mileage"].median(),
        "engine": rec["engine"].median(),
        "max_power": rec["max_power"].median(),
        "price_lakh": budget
    }])

    # --------------------------------------------------
    # 3. Transform user preferences
    # --------------------------------------------------
    user_encoded = encoder.transform(
        user_preferences[categorical]
    )

    user_scaled = scaler.transform(
        user_preferences[numerical]
    )

    user_vector = np.hstack([
        user_encoded,
        user_scaled
    ])

    # --------------------------------------------------
    # 4. Cosine similarity
    # --------------------------------------------------
    user_similarity = cosine_similarity(
        user_vector,
        matrix
    )[0]

    candidates["similarity"] = user_similarity[candidates.index]

    # --------------------------------------------------
    # 5. Explicit preference matching
    # --------------------------------------------------
    candidates["preference_match"] = (
        (candidates["brand"].str.upper() == brand.upper()).astype(int) +
        (candidates["body_type"].str.upper() == body_type.upper()).astype(int) +
        (candidates["transmission"].str.upper() == transmission.upper()).astype(int) +
        (candidates["city"].str.lower() == city.lower()).astype(int)
    ) / 4

    # --------------------------------------------------
    # 6. K-Means cluster
    # --------------------------------------------------
    user_cluster = final_kmeans.predict(user_vector)[0]

    candidates["cluster_match"] = (
        candidates["cluster"] == user_cluster
    ).astype(int)

    # --------------------------------------------------
    # 7. Final hybrid score
    # --------------------------------------------------
    candidates["hybrid_score"] = (
        0.70 * candidates["preference_match"] +
        0.25 * candidates["similarity"] +
        0.05 * candidates["cluster_match"]
    )

    candidates = candidates.sort_values(
        by="hybrid_score",
        ascending=False
    )

    # --------------------------------------------------
    # 8. Get additional details from original dataset
    # --------------------------------------------------
    result = df.loc[candidates.head(top_n).index].copy()

    # Add recommendation scores
    result["preference_match"] = candidates.loc[
        result.index, "preference_match"
    ]

    result["similarity"] = candidates.loc[
        result.index, "similarity"
    ]

    result["cluster_match"] = candidates.loc[
        result.index, "cluster_match"
    ]

    result["hybrid_score"] = candidates.loc[
        result.index, "hybrid_score"
    ]

    # --------------------------------------------------
    # 9. Final display columns
    # --------------------------------------------------
    display_columns = [
        "brand",
        "model",
        "variant",
        "fuel_type",
        "body_type",
        "transmission",
        "city",
        "km_driven",
        "model_year",
        "price_lakh",
        "preference_match",
        "similarity",
        "cluster_match",
        "hybrid_score"
    ]

    return result[display_columns]

In [18]:
recommendations = recommend_cars(
    budget=10,
    fuel="PETROL",
    brand="Hyundai",
    body_type="SUV",
    transmission="Automatic",
    city="chennai",
    top_n=10
)

recommendations

,brand,model,variant,fuel_type,body_type,transmission,city,km_driven,model_year,price_lakh,preference_match,similarity,cluster_match,hybrid_score
2696,Hyundai,Hyundai Creta,1.6 VTVT AT SX Plus,PETROL,SUV,Automatic,chennai,27370,2016,9.64,1.00,0.704139,0,0.876035
2653,Hyundai,Hyundai Creta,1.6 VTVT AT SX Plus,PETROL,SUV,Automatic,chennai,85781,2016,9.16,1.00,0.697127,0,0.874282
2701,Hyundai,Hyundai i20,Petrol CVT Asta,PETROL,Hatchback,Automatic,chennai,39323,2018,6.86,0.75,0.730757,1,0.757689
2611,Hyundai,Hyundai Grand i10,AT Asta,PETROL,Hatchback,Automatic,chennai,42856,2016,5.20,0.75,0.728476,1,0.757119
2188,Hyundai,Hyundai Grand i10,AT Sportz,PETROL,Hatchback,Automatic,chennai,18837,2018,6.50,0.75,0.728298,1,0.757075
2377,Hyundai,Hyundai Xcent,1.2 Kappa S AT,PETROL,Sedan,Automatic,chennai,64011,2016,5.25,0.75,0.727193,1,0.756798
2215,Hyundai,Hyundai i20,Sportz Plus CVT,PETROL,Hatchback,Automatic,chennai,35000,2019,8.00,0.75,0.723422,1,0.755856
2550,Hyundai,Hyundai Grand i10,1.2 Kappa Magna AT,PETROL,Hatchback,Automatic,chennai,17797,2016,4.95,0.75,0.717845,1,0.754461
2018,Hyundai,Hyundai Grand i10 Nios,AMT Sportz,PETROL,Hatchback,Automatic,chennai,27749,2019,6.65,0.75,0.712590,1,0.753148
2407,Hyundai,Hyundai i20,Asta Option CVT BSIV,PETROL,Hatchback,Automatic,chennai,10000,2019,8.50,0.75,0.712476,1,0.753119


In [19]:
import joblib

recommendation_artifacts = {
    "encoder": encoder,
    "scaler": scaler,
    "recommendation_matrix": matrix,
    "rec": rec,
    "final_kmeans": final_kmeans,
    "categorical_features": categorical,
    "numerical_features": numerical
}

joblib.dump(
    recommendation_artifacts,
    "recommendation_artifacts.joblib"
)

['recommendation_artifacts.joblib']

In [20]:
df.to_csv(
    "cleaned_cars.csv",
    index=False
)